# Receipt extraction with a fine-tuned small VLM: train + evaluate on a free T4

This notebook produces every GPU number in the README / reports:

1. zero-shot evaluation of `Qwen2.5-VL-3B-Instruct` on the CORD v2 **test** split
2. QLoRA fine-tuning on CORD v2 **train** (validation split for eval loss)
3. fine-tuned evaluation on the same test split, same prompt, same pipeline
4. robustness runs (rotation, blur, JPEG, small fonts, multi-page) and a resolution ablation
5. the API with the real model + the 5-parallel-request concurrency test
6. `reports/evaluation_report.md`

Runtime: *Runtime → Change runtime type → T4 GPU*. Total time is roughly 3–4 h; every step writes its results to disk, so a disconnect only costs the current step (training resumes from checkpoints with `--resume`).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144 MiB


In [2]:
REPO_URL = "https://github.com/mean-machine-18/document_extraction_service"   # <- set
HF_ADAPTER_REPO = "https://huggingface.co/meanmachine18/qwen25vl3b-cord-lora"

USE_DRIVE = False

from pathlib import Path

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "Document Retrieval App" / "requirements-train.txt").exists():
            return p
    return None

repo = find_repo_root()

if repo is None:  # not inside the project (e.g. fresh Colab runtime), so clone it
    repo = Path("/content/receipt-vlm-extraction")
    if not repo.exists():
        !git clone $REPO_URL {str(repo)}

%cd {str(repo)}
!pip install -q -r "Document Retrieval App/requirements-train.txt"

C:\Users\sayan\PycharmProjects\document_extraction_service



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
OUT = "outputs/qwen25vl3b-cord-lora"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/receipt-vlm/qwen25vl3b-cord-lora"
    os.makedirs("/content/drive/MyDrive/receipt-vlm/results", exist_ok=True)
    !rm -rf results && ln -s /content/drive/MyDrive/receipt-vlm/results results
ADAPTER = f"{OUT}/adapter"
print(OUT)

outputs/qwen25vl3b-cord-lora


In [10]:
# Hugging Face token with write access (only needed to push the adapter)
from huggingface_hub import notebook_login
notebook_login()

## 0. Grounding audit (CPU only, no model)

In [16]:
%cd "Document Retrieval App"
import os; os.makedirs("results", exist_ok=True)

!python -m scripts.grounding_audit --split test --threshold 85 --out results/grounding_audit.json
!python -m scripts.grounding_audit --split test --threshold 75 --out results/grounding_audit_t75.json
!python -m scripts.grounding_audit --split test --threshold 95 --out results/grounding_audit_t95.json

!python -m scripts.evaluate --oracle --name oracle_upper_bound

C:\Users\sayan\PycharmProjects\document_extraction_service\Document Retrieval App\notebooks\receipt-vlm-extraction\receipt-vlm-extraction\receipt-vlm-extraction\receipt-vlm-extraction\Document Retrieval App
{
  "split": "test",
  "documents": 100,
  "text_match_threshold": 85.0,
  "gold_verified_rate": {
    "line_items.amount": {
      "checked": 266,
      "verified": 248,
      "rate": 0.9323308270676691
    },
    "line_items.name": {
      "checked": 287,
      "verified": 265,
      "rate": 0.9233449477351916
    },
    "line_items.qty": {
      "checked": 237,
      "verified": 162,
      "rate": 0.6835443037974683
    },
    "line_items.unit_price": {
      "checked": 67,
      "verified": 56,
      "rate": 0.835820895522388
    },
    "subtotal": {
      "checked": 65,
      "verified": 64,
      "rate": 0.9846153846153847
    },
    "tax": {
      "checked": 41,
      "verified": 37,
      "rate": 0.9024390243902439
    },
    "total": {
      "checked": 95,
      "verified":

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not in

{
  "split": "test",
  "documents": 100,
  "text_match_threshold": 75.0,
  "gold_verified_rate": {
    "line_items.amount": {
      "checked": 266,
      "verified": 250,
      "rate": 0.9398496240601504
    },
    "line_items.name": {
      "checked": 287,
      "verified": 274,
      "rate": 0.9547038327526133
    },
    "line_items.qty": {
      "checked": 237,
      "verified": 168,
      "rate": 0.7088607594936709
    },
    "line_items.unit_price": {
      "checked": 67,
      "verified": 56,
      "rate": 0.835820895522388
    },
    "subtotal": {
      "checked": 65,
      "verified": 64,
      "rate": 0.9846153846153847
    },
    "tax": {
      "checked": 41,
      "verified": 37,
      "rate": 0.9024390243902439
    },
    "total": {
      "checked": 95,
      "verified": 90,
      "rate": 0.9473684210526315
    }
  },
  "hallucination_detection_rate": {
    "line_items.amount": {
      "injected": 85,
      "caught": 83,
      "rate": 0.9764705882352941
    },
    "line_ite


100%|██████████| 100/100 [01:26<00:00,  1.15it/s]


{
  "split": "test",
  "documents": 100,
  "text_match_threshold": 95.0,
  "gold_verified_rate": {
    "line_items.amount": {
      "checked": 266,
      "verified": 247,
      "rate": 0.9285714285714286
    },
    "line_items.name": {
      "checked": 287,
      "verified": 226,
      "rate": 0.7874564459930313
    },
    "line_items.qty": {
      "checked": 237,
      "verified": 145,
      "rate": 0.6118143459915611
    },
    "line_items.unit_price": {
      "checked": 67,
      "verified": 56,
      "rate": 0.835820895522388
    },
    "subtotal": {
      "checked": 65,
      "verified": 64,
      "rate": 0.9846153846153847
    },
    "tax": {
      "checked": 41,
      "verified": 37,
      "rate": 0.9024390243902439
    },
    "total": {
      "checked": 95,
      "verified": 90,
      "rate": 0.9473684210526315
    }
  },
  "hallucination_detection_rate": {
    "line_items.amount": {
      "injected": 85,
      "caught": 83,
      "rate": 0.9764705882352941
    },
    "line_ite


100%|██████████| 100/100 [01:23<00:00,  1.19it/s]


{
  "micro_f1_raw": 1.0,
  "micro_f1_grounded": 0.9220580743759552,
  "hallucination": 0.012287334593572778,
  "hallucination_grounded": 0.0033149171270718232,
  "latency": {
    "vlm_per_page_p50_s": 0.0,
    "vlm_per_page_p95_s": 0.0,
    "ocr_per_doc_p50_s": 0.7029999999999745,
    "end_to_end_per_doc_p50_s": 0.7809999999999491,
    "end_to_end_per_doc_p95_s": 1.866049999999927,
    "output_tokens_p50": 0.0
  },
  "peak_gpu_memory_mb": null,
  "json": {
    "strict_valid_rate": 1.0,
    "repaired_rate": 0.0,
    "unparseable_rate": 0.0,
    "service_valid_rate": 1.0,
    "timed_out_pages": 0
  }
}



oracle_upper_bound: 100%|██████████| 100/100 [01:29<00:00,  1.12it/s]


## 1. Zero-shot baseline

In [5]:
%cd "Document Retrieval App"
!python -m scripts.evaluate --name zeroshot

/Users/sayantan/Projects/Document Retrieval App/.venv/bin/python: Error while finding module specification for 'scripts.evaluate' (ModuleNotFoundError: No module named 'scripts')


## 2. QLoRA fine-tuning
2 epochs over 800 receipts at batch 1 × grad-accum 8 = 200 optimizer steps. Re-run the same cell with `--resume` after a disconnect.

In [18]:
!python -m training.train_lora --output-dir "$OUT" --epochs 2 --max-visual-tokens 1024 --push-to-hub "$HF_ADAPTER_REPO"

Traceback (most recent call last):
  File "C:\Users\sayan\PycharmProjects\document_extraction_service\.venv\Lib\site-packages\transformers\utils\import_utils.py", line 1967, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\sayan\AppData\Local\Python\pythoncore-3.12-64\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_re

## 3. Fine-tuned evaluation (same split, prompt and pipeline as zero-shot)

In [ ]:
!python -m scripts.evaluate --name finetuned --adapter "$ADAPTER"

## 4. Failure modes and resolution trade-off

In [ ]:
for p in ["rotate5", "rotate90", "blur", "jpeg", "downscale", "multipage"]:
    !python -m scripts.evaluate --name finetuned_{p} --adapter "$ADAPTER" --perturb {p} | tail -3

In [ ]:
for vt in [256, 512, 1024, 1536]:
    !python -m scripts.evaluate --name finetuned_vt{vt} --adapter "$ADAPTER" --max-visual-tokens {vt} | tail -3

## 5. API with the real model + concurrency test
Starts uvicorn in the background with the fine-tuned adapter and fires 5 simultaneous requests.

In [ ]:
!python -m scripts.make_sample_pdfs --n 3 --out samples
import subprocess, time, requests
env = dict(os.environ, ADAPTER=ADAPTER, MAX_QUEUE="3", QUEUE_TIMEOUT_S="60", JOB_TIMEOUT_S="90")
server = subprocess.Popen(["uvicorn", "docextract.service.app:app", "--port", "8000", "--workers", "1"],
                          env=env, stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        if requests.get("http://localhost:8000/readyz", timeout=2).ok: break
    except Exception: pass
    time.sleep(2)
print(requests.get("http://localhost:8000/readyz").json())

In [ ]:
!curl -s -F file=@samples/cord_test_000.pdf localhost:8000/extract | python -m json.tool | head -60

In [ ]:
!python -m scripts.concurrency_test --url http://localhost:8000 --pdf samples/cord_test_000.pdf --n 5 --save reports/concurrency_output.txt

In [ ]:
# bad-input behaviour against the real server
!head -c 100 /dev/urandom > bad.pdf && curl -s -F file=@bad.pdf localhost:8000/extract; echo
!printf '%%PDF-1.7 broken' > corrupt.pdf && curl -s -F file=@corrupt.pdf localhost:8000/extract; echo
!python -c "import fitz; d=fitz.open(); d.new_page(); d.save('locked.pdf', encryption=fitz.PDF_ENCRYPT_AES_256, user_pw='x', owner_pw='y')" && curl -s -F file=@locked.pdf localhost:8000/extract; echo
server.terminate()

## 6. Report

In [ ]:
!python -m scripts.make_report --results results --train-metrics "$OUT/train_metrics.json"
!cp "$OUT/train_metrics.json" results/ 2>/dev/null; zip -qr artifacts.zip results reports
from google.colab import files; files.download("artifacts.zip")